# ViTrueVoice — Pipeline Huấn Luyện SSL-AASIST & Đánh Giá Đối Chứng H1/H2 Trên Kaggle GPU

**Mục tiêu thí nghiệm (Must #2 - W5)**:
1. **Huấn luyện 3 nhánh thí nghiệm**:
   - **Nhánh A (Không tăng cường)**: `mode="clean"` (10.699 mẫu), không transform, Phase 1 = 4 epochs, Phase 2 = 60 epochs (nhân 3 để cân bằng optimizer steps).
   - **Nhánh B0 (Chỉ nén Codec, không RawBoost)**: `mode="augmented"` (31.327 mẫu), không transform, Phase 1 = 4 epochs, Phase 2 = 20 epochs.
   - **Nhánh B (ViTrueVoice Đầy đủ: Codec + RawBoost)**: `mode="augmented"` (31.327 mẫu), `make_rawboost()`, Phase 1 = 4 epochs, Phase 2 = 20 epochs.
2. **Đánh giá tự động trên Test In-domain và Test OOD** qua `scripts/eval_ssl_aasist.py`.
3. **Kiểm chứng giả thuyết khoa học** qua `scripts/compare_h1.py`:
   - **H1**: Giảm $\ge 30\%$ EER tương đối trên kênh nén (G.711 A-law, AMR-NB) so với Mô hình A.
   - **H2**: EER $\le 15\%$ trên công cụ TTS chưa từng thấy (`VieNeu-TTS`).
   - **Phân rã biến**: So sánh B0 vs B để bóc tách chính xác đóng góp của RawBoost.


In [ ]:
# 1. Cài đặt phụ thuộc & Kiểm tra GPU
import subprocess
import sys
import torch

print('📦 Đang kiểm tra & cài đặt thư viện phụ thuộc...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'soundfile', 'librosa', 'transformers'], check=False)

print('=' * 60)
print(f'PyTorch Version : {torch.__version__}')
print(f'CUDA Available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('⚠️ CẢNH BÁO: Không có GPU! Huấn luyện SSL-AASIST trên CPU sẽ rất chậm.')
print('=' * 60)


In [ ]:
# 2. Khởi tạo mã nguồn ViTrueVoice trên Kaggle container
import base64
import zlib
import json
import sys
from pathlib import Path

PKG_B64 = '...'

pkg_data = json.loads(zlib.decompress(base64.b64decode(PKG_B64.encode('ascii'))).decode('utf-8'))
working_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')

for rel_path, content in pkg_data.items():
    file_path = working_dir / rel_path
    file_path.parent.mkdir(parents=True, exist_ok=True)
    file_path.write_text(content, encoding='utf-8')

if str(working_dir) not in sys.path:
    sys.path.insert(0, str(working_dir))

print(f'✅ Đã khởi tạo thành công {len(pkg_data)} files mã nguồn tại: {working_dir}')


In [ ]:
# 3. Định vị dữ liệu âm thanh và các tệp Manifest
import zipfile
from pathlib import Path

working_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')

# Kiểm tra nếu dữ liệu đã tồn tại sẵn
candidates = [
    working_dir / 'data',
    Path('/kaggle/input/vietrue-model-training/data'),
    Path('/kaggle/input/vitruevoice-data-processed/data'),
    Path('/kaggle/input/vietruevoice-processed/data'),
    Path('./data'),
]

data_dir = None
for c in candidates:
    if c.exists() and (c / 'manifests' / 'train.csv').exists():
        data_dir = c
        break

if not data_dir:
    # Tìm file ZIP đã nén từ Phase 1
    zip_candidates = (
        list(Path('/kaggle/input').rglob('*vitruevoice*.zip')) +
        list(working_dir.glob('*vitruevoice*.zip')) +
        list(Path('out').glob('*vitruevoice*.zip'))
    )
    if zip_candidates:
        zip_path = zip_candidates[0]
        print(f'📦 Tìm thấy file nén dữ liệu: {zip_path}. Đang giải nén...')
        target_dir = working_dir / 'data'
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(working_dir)
        data_dir = target_dir
        print(f'✅ Đã giải nén vào {data_dir}')

if data_dir and (data_dir / 'manifests' / 'train.csv').exists():
    manifest_dir = data_dir / 'manifests'
    audio_root = working_dir
else:
    # Fallback kiểm tra thư mục manifests trực tiếp
    manifest_dir = Path('data/manifests') if Path('data/manifests/train.csv').exists() else Path('out')
    audio_root = Path('.')

print(f'📂 Manifest Directory: {manifest_dir}')
print(f'🎵 Audio Root        : {audio_root}')
assert (manifest_dir / 'train.csv').exists(), f'❌ Không tìm thấy train.csv tại {manifest_dir}!'
assert (manifest_dir / 'dev.csv').exists(), f'❌ Không tìm thấy dev.csv tại {manifest_dir}!'
assert (manifest_dir / 'test_indomain.csv').exists(), f'❌ Không tìm thấy test_indomain.csv tại {manifest_dir}!'
print('✅ Đã xác thực đầy đủ các tệp manifest bắt buộc.')


In [ ]:
# 4. Huấn luyện Mô hình A (Không tăng cường - Baseline SSL-AASIST)
# mode='clean' (10.699 mẫu sạch), transform=None, Phase 1 = 4 ep, Phase 2 = 60 ep
!python scripts/train.py \
    --branch A \
    --manifest-dir "{manifest_dir}" \
    --audio-root "{audio_root}" \
    --out-dir out/run_A \
    --batch-size 16


In [ ]:
# 5. Huấn luyện Mô hình B0 (Chỉ nén Codec, không RawBoost)
# mode='augmented' (31.327 mẫu), transform=None, Phase 1 = 4 ep, Phase 2 = 20 ep
!python scripts/train.py \
    --branch B0 \
    --manifest-dir "{manifest_dir}" \
    --audio-root "{audio_root}" \
    --out-dir out/run_B0 \
    --batch-size 16


In [ ]:
# 6. Huấn luyện Mô hình B (ViTrueVoice Đầy Đủ: Codec + RawBoost Online)
# mode='augmented' (31.327 mẫu), transform=make_rawboost(), Phase 1 = 4 ep, Phase 2 = 20 ep
!python scripts/train.py \
    --branch B \
    --manifest-dir "{manifest_dir}" \
    --audio-root "{audio_root}" \
    --out-dir out/run_B \
    --batch-size 16


In [ ]:
# 7. Đánh giá tự động các mô hình trên Test Sets (In-domain & OOD)
import os

manifest_args = f"in_domain={manifest_dir}/test_indomain.csv"
if (manifest_dir / "test_ood.csv").exists():
    manifest_args += f" ood={manifest_dir}/test_ood.csv"

for branch in ["A", "B0", "B"]:
    ckpt = f"out/run_{branch}/best_model.pt"
    if os.path.exists(ckpt):
        print(f"\n{'='*60}\n🔍 Đang chấm điểm Model {branch} ({ckpt})...\n{'='*60}")
        !python scripts/eval_ssl_aasist.py \
            --ckpt "{ckpt}" \
            --manifest {manifest_args} \
            --audio-root "{audio_root}" \
            --role eval \
            --out "out/eval_metrics_{branch}.csv" \
            --scores-out "out/scores_{branch}.csv"
    else:
        print(f"⚠️ Checkpoint không tồn tại: {ckpt}")


In [ ]:
# 8. Kiểm chứng giả thuyết khoa học H1 & H2
import os

if os.path.exists("out/eval_metrics_A.csv") and os.path.exists("out/eval_metrics_B.csv"):
    print('=' * 75)
    print('📊 ĐỐI CHỨNG GIẢ THUYẾT H1 & H2 (MÔ HÌNH A vs MÔ HÌNH B):')
    print('=' * 75)
    !python scripts/compare_h1.py \
        --a out/eval_metrics_A.csv \
        --b out/eval_metrics_B.csv \
        --out-dir out/comparison_A_B

if os.path.exists("out/eval_metrics_B0.csv") and os.path.exists("out/eval_metrics_B.csv"):
    print('\n' + '=' * 75)
    print('📊 ĐỐI CHỨNG BÓC TÁCH ĐÓNG GÓP RAWBOOST (MÔ HÌNH B0 vs MÔ HÌNH B):')
    print('=' * 75)
    !python scripts/compare_h1.py \
        --a out/eval_metrics_B0.csv \
        --b out/eval_metrics_B.csv \
        --out-dir out/comparison_B0_B


In [ ]:
# 9. Hiển thị báo cáo tóm tắt Markdown trên Notebook
from pathlib import Path
from IPython.display import display, Markdown

rep = Path('out/comparison_A_B/h1_h2_summary.md')
if rep.exists():
    display(Markdown(rep.read_text(encoding='utf-8')))
